# အခန်း ၁၁ - Agent-to-Agent (A2A) Protocol


## ပြင်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2A ပရိုတိုကောဆိုတာဘာလဲ?

**Agent-to-Agent (A2A) ပရိုတိုကော** သည် AI အေးဂျင့်များအကြား ဆက်သွယ်ဆောင်ရွက်နိုင်ရန်၊
တစ်ဦးနှင့်တစ်ဦး ရှာဖွေရေးနှင့် ပူးပေါင်းဆောင်ရွက်နိုင်စေရန် ဖွင့်လှစ်ထားသော စံနှုန်းတစ်ခုဖြစ်ပြီး၊
ကွဲပြားသောဖရိမ်ဝတ်များတွင် တည်ဆောက်ထားခြင်း သို့မဟုတ် ကွဲပြားသော ၀န်ဆောင်မှုများဖြင့် ဖန်တီးထားခြင်း များပင်ဖြစ်ပါက




- **ရှာဖွေမှု (Discovery)** – အေးဂျင့်များသည် မိမိစွမ်းရည်များအား ဖော်ပြသည့် *Agent Card* ကိုထုတ်ပြန်ပြီး၊
  တခြားအေးဂျင့်များ (သို့မဟုတ် အော်ကပ်စတရေတာများ) အတွက် တာဝန်အတွက် သင့်တော်သော အထူးပြု အသုံးပြုသူများ ရှာဖွေရှာရန် လွယ်ကူစေသည်။
- **စာတိုက်ပို့ခြင်း (Message Passing)** – အေးဂျင့်များသည် ပုံသေစနစ်တကျရှိသည့် စာတိုက်များကို ပရိုတိုကောတစ်ခုတည်းဖြင့် လဲလှယ်ချက်ပြုလုပ်သည်။
  ထိုကွောင့္ တစ်ဦးမှတစ်ဦး သို့ မေးခွန်းတစ်ခု ပေးပို့၍လည်း ထိုအေးဂျင့်၏ အတွင်းပိုင်း အကောင်အထည်ဖော်မှုမည်သို့ ဖြစ်စေ မည်သူမဆို နားလည် တုံ့ပြန်နိုင်သည်။
- **တာဝန်သက်တမ်း (Task Lifecycle)** – A2A သည် *တင်သွင်းထားသည်၊* *အလုပ်လုပ်နေသည်၊* *ပြီးစီးသည်၊* နှင့် *မအောင်မြင်ပါ* စသည်တို့ကို သတ်မှတ်သည်။
  အော်ကပ်စတရေတာအား တာဝန်ကို တာဝန်ပေးထားသည့် အဆင့်မြောက်မှုကို ပြည့်စုံသိရှိနိုင်သည်။

ဤသင်ခန်းစာတွင် A2A ပုံစံဖြင့် သီးခြားအထူးပြု ခရီးသွားအေးဂျင့်သုံးခုကို
အလုပ်စဉ်တစ်ခုတွင်ချိတ်ဆက်ပြီး အေးဂျင့်တိုင်းသည် မိမိ၏ ကျွမ်းကျင်မှုကို ထောက်ပံ့ကူညီကာ နောက်တစ်ဦးထံအောင်မြင်မှုများ ပို့ဆောင်ပေးသည်ကို သရုပ်ပြထားသည်။


## အထူးပြု ခရီးသွားခေါင်းဆောင် Agents များ ဖန်တီးခြင်း


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## အလုပ်တိုက်ဆောင်ရွက်မှုမှတဆင့် အမျိုးမျိုးသော အေးဂျင့်များအကြား ပူးပေါင်းဆောင်ရွက်ခြင်း

ကျွန်ုပ်တို့သည် သုံးဦးသော အေးဂျင့်များကို A2A ဆက်သွယ်မှုကို လိုက်စားသလို တန်းစီ စီမံခန့်ခွဲပုံစနစ်သို့ ချိတ်ဆက်ပေးသည်။

1. **CurrencyExchangeAgent** သည် အသုံးပြုသူ၏ မေးမြန်းချက်ကို လက်ခံပြီး ငွေကြေး လမ်းညွှန်ချက်များ ထုတ်ပေးသည်။
2. **ActivityPlannerAgent** သည် တိုးတက်လာသော နောက်ခံအချက်အလက်ကို လက်ခံပြီး လှုပ်ရှားမှု အကြံပေးချက်များ ထည့်သွင်းပေးသည်။
3. **TravelManagerAgent** သည် နှစ်ဖက်လက်ခံရရှိသော အချက်အလက်များကို စုပေါင်းပြီး နောက်ဆုံး ခရီးသွား အကျဉ်းချုပ် တစ်ခု ချမှတ်ပေးသည်။


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ထုတ်လုပ်မှုတွင် A2A ကိုနားလည်ခြင်း

ထုတ်လုပ်မှုပတ်ဝန်းကျင်တွင် A2A protocols သည် အင်အားပြင်းသော cross-service ရှေ့ဆောင်များကို ဖြေရှင်းပေးသည်။

| လူ့စွမ်းရည် | ဖော်ပြချက် |
|---|---|
| **Cross-framework interop** | တစ်ခုသော framework ဖြင့် တည်ဆောက်ထားသော အေးဂျင့်သည် A2A ကိုက်ညီသော အခြား framework များအားလုံးနှင့် လုပ်ဆောင်မှုများကို delegate လုပ်နိုင်ပြီး၊ တကယ့် အဖွဲ့အစည်းများအကြား အပြန်အလှန် လုပ်ဆောင်နိုင်မှုကို ဖော်ဆောင်ပေးသည်။ |
| **Service boundaries** | အေးဂျင့်များသည် အထူး microservices, cloud ဒေသများသို့မဟုတ် သီးခြား အဖွဲ့အစည်းများတွင်တည်နေထိုင်နိုင်ပြီး၊ seamless အလုပ်လုပ်နိုင်ပါသည်။ |
| **Dynamic discovery** | Orchestrator တစ်ခုသည် runtime တွင် Agent Card မှတ်တမ်းစာရင်းကို query လုပ်၍ အကောင်းဆုံး specialist ကို ရှာဖွေနိုင်သည်။ |
| **Streaming & push notifications** | A2A သည် Server-Sent Events (SSE) ကို ထောက်ပံ့ကူညီပြီး သက်တမ်းရှည်သော လုပ်ငန်းများအတွက် push notification များကိုပေးစွမ်းနိုင်သည်။ |

ကျွန်ုပ်တို့ အထက်တွင် တည်ဆောက်ထားသော workflow သည် ဒီပုံစံ၏ ရိုးရိုးရှင်းရှင်းနည်းဖြင့် ထုတ်လုပ်မှုတွင် လူသုံးလုပ်ဆောင်မှု ဖြစ်သည်။ အမှန်တကယ် အဖွဲ့အစည်း တစ်ခုစီသည် HTTP endpoint ထုတ်လွှင့်၊ Agent Card တစ်ခုထုတ်ပြသပြီး A2A JSON-RPC protocol ဖြင့် ဆက်သွယ်ဆောင်ရွက်သည်။ 



## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင်သင်ယူခဲ့သည့်အရာများမှာ -

1. **A2A ပရိုတိုကောသည် ဘာလဲ** — agent-to-agent ရှာဖွေခြင်း၊ မက်ဆေ့ခ်ျပို့ခြင်းနှင့် တာဝန်စီမံခန့်ခွဲမှုအတွက် ဖွင့်လင်းစံချိန်တစ်ခု။
   ဝှက်ချက်။
2. **ပစ္စည်းပြုလုပ်ထားသော agent များကို ဘယ်လို ဖန်တီးမလဲ** — ငွေလဲလှယ်မှု agent၊ လှုပ်ရှားမှု စီစဉ်သူ agent နှင့် ခရီးသွား စီမံခန့်ခွဲသူ orchestrator တို့။

3. **Agent များကို workflow တစ်ခုထဲသို့ ဘယ်လို တပ်ဆင်မလဲ** — agent များအကြား အဆက်မပြတ် မက်ဆေ့ခ်ျပို့ဆက်သွယ်မှု မော်ဒယ်ဖော်နိုင်ရန် `WorkflowBuilder` ကိုအသုံးပြုခြင်း။

4. **A2A ကို ထုတ်လုပ်ရေးအတွက် ဘယ်လို အသုံးပြုသလဲ** — dynamic ရှာဖွေမှုနှင့် streaming update များဖြင့် cross-framework၊ cross-service ညှိနှိုင်း ဆောင်ရွက်နိုင်ခြင်း။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
